# Credit Risk Scoring
Un parcours pour comprendre puis reproduire un prototype académique. Lire README.md avant de modifier le modèle. Les résultats fournis sont une première exécution, pas une promesse de performance.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib

## 1. Retrouver le dossier
Sur Colab, importer le dossier credit_risk_project dans /content. En local, ouvrir le notebook depuis le dossier du projet.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd()
if not (root / 'pipeline.py').exists():
    root = root / 'credit_risk_project'
assert (root / 'pipeline.py').exists(), 'Placer le dossier credit_risk_project dans le répertoire courant.'
os.chdir(root)
sys.path.insert(0, str(root.resolve()))
import pandas as pd
import pipeline

## 2. Comprendre les données
Que représente une ligne ? Quelle est la cible ? Les montants sont en dollars taïwanais. Formule tes réponses avant de poursuivre.

In [ ]:
df = pipeline.load_data()
display(df.head())
print('Dimensions :', df.shape)
print('Taux de défaut :', df[pipeline.TARGET].mean())
display(df.isna().sum().to_frame('manquants'))

## 3. Examiner les codes et profils dupliqués
Ne pas supprimer automatiquement : plusieurs clients peuvent partager les mêmes caractéristiques. Que deviennent les codes inhabituels dans prepare() ?

In [ ]:
for col in ['EDUCATION','MARRIAGE','PAY_0']:
    print(col, df[col].value_counts().sort_index().to_dict())
print('Doublons hors ID :', df.drop(columns='ID').duplicated().sum())
display(df.groupby('PAY_0')[pipeline.TARGET].agg(['count','mean']))

## 4. Lire puis exécuter le pipeline
Avant de lancer, repère dans pipeline.py : les trois splits, les pipelines de prétraitement, la calibration interne et la sélection du seuil. Pourquoi ne règle-t-on pas le seuil sur le test ?

In [ ]:
result = pipeline.main()

## 5. Comparer les résultats
Average Precision se compare au taux de défaut. ROC-AUC mesure le classement ; log-loss et Brier évaluent les probabilités sous plusieurs dimensions. La courbe de calibration aide à détecter les écarts.

In [ ]:
display(pd.read_csv('outputs/validation_metrics.csv'))
display(pd.read_csv('outputs/test_metrics.csv'))
from IPython.display import Image, display
display(Image(filename='outputs/evaluation.png'))

## 6. Explorer les erreurs et utiliser SQL
Quels faux négatifs le modèle manque-t-il ? Les facteurs les plus prédictifs sont-ils nécessairement des causes ?

In [ ]:
display(pd.read_csv('outputs/feature_importance.csv').head(10))
import sqlite3
with sqlite3.connect('outputs/credit_risk.sqlite') as con:
    display(pd.read_sql_query('SELECT risk_band, COUNT(*) AS n, AVG(actual_default) AS default_rate FROM test_predictions GROUP BY risk_band', con))

## 7. Construire et présenter
Suis POWER_BI.md. Puis écris cinq phrases : question métier, données, protocole, résultat chiffré, limite principale. Évite de relancer des réglages après lecture du test : toute itération guidée par ce test en compromet l'indépendance.